# Survival Analysis: Latent vs CMR Risk Scores

Two-step pipeline:

1. **Risk-score derivation on train split**
   - Fit a Cox model on latent features to derive a scalar latent risk score.
   - Fit a Cox model on the 7 CMR markers + auxiliary variables `[age, sex, bsa]` to derive a scalar CMR risk score.

2. **Downstream survival models with penalizer tuning**
   - Add either score to PCP-HF covariates.
   - Fit Cox PH on train with CV-based penalizer tuning.
   - Evaluate on test with:
     - C-index
     - AUROC at 3 and 5 years
     - Expected/Observed (E/O) ratio at 3 and 5 years.

In [ ]:
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceWarning

from cardiac_latent_ode.utils.survival_analysis import (
    compute_auroc_at_time_stratified,
    concordance_stratified,
    expected_observed_ratio_at_time_stratified,
    load_auxiliary_features,
    load_cmr_marker_features,
    load_latent_features,
    load_pcp_hf_features,
    prepare_survival_targets,
    spline_expand_test,
    spline_expand_train,
    tune_cox_penalizer,
)

warnings.filterwarnings("ignore", category=ConvergenceWarning)


# -----------------------------------------------------------------------------
# Config: set data paths and endpoint here
# -----------------------------------------------------------------------------
cohort_path = Path(os.getenv("DATA_DIR")) / "preprocessed" / "cohort.csv"
outcomes_path = Path(os.getenv("DATA_DIR")) / "preprocessed" / "outcomes.csv"
covariates_path = Path(os.getenv("DATA_DIR")) / "preprocessed" / "demographics.csv"
cmr_markers_path = Path(os.getenv("DATA_DIR")) / "preprocessed" / "clinical_markers.csv"
latents_path = Path(os.getenv("SAVE_DIR")) / "latent" / "latents.npz"

endpoint_col = "heart_failure"  # Change to different endpoint if wanted
seed = 0
n_splits = 5
penalizer_grid = [1e-3, 1e-2, 1e-1, 1.0, 10.0]
time_3yr = 3.0 * 365.25
time_5yr = 5.0 * 365.25


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def _fit_cox(
    df_train: pd.DataFrame,
    feature_cols: list[str],
    penalizer: float,
    strata_arr: np.ndarray | None = None,
) -> CoxPHFitter:
    train_df = df_train[feature_cols + ["time", "event"]].copy()

    cph = CoxPHFitter(penalizer=penalizer)
    if strata_arr is None:
        cph.fit(train_df, duration_col="time", event_col="event")
    else:
        train_df["_strata"] = strata_arr
        cph.fit(train_df, duration_col="time", event_col="event", strata=["_strata"])
    return cph


def _predict_partial_risk(
    cph: CoxPHFitter,
    df: pd.DataFrame,
    feature_cols: list[str],
    strata_arr: np.ndarray | None = None,
) -> np.ndarray:
    X = df[feature_cols].copy()
    if strata_arr is not None:
        X["_strata"] = strata_arr
    return cph.predict_partial_hazard(X).to_numpy(np.float64)


def _predict_event_prob(
    cph: CoxPHFitter,
    df: pd.DataFrame,
    feature_cols: list[str],
    horizon_days: float,
    strata_arr: np.ndarray | None = None,
) -> np.ndarray:
    X = df[feature_cols].copy()
    if strata_arr is not None:
        X["_strata"] = strata_arr

    ref_idx = X.index
    surv = cph.predict_survival_function(X, times=[horizon_days]).T.reindex(ref_idx).values.flatten()
    return 1.0 - np.clip(surv, 0.0, 1.0)


def _evaluate_model(
    cph: CoxPHFitter,
    df_test: pd.DataFrame,
    feature_cols: list[str],
    time_3yr: float,
    time_5yr: float,
    strata_arr: np.ndarray,
) -> dict:
    t_te = df_test["time"].to_numpy(np.float64)
    e_te = df_test["event"].to_numpy(np.int64)

    risk = _predict_partial_risk(cph, df_test, feature_cols, strata_arr=strata_arr)
    ep3 = _predict_event_prob(cph, df_test, feature_cols, time_3yr, strata_arr=strata_arr)
    ep5 = _predict_event_prob(cph, df_test, feature_cols, time_5yr, strata_arr=strata_arr)

    ci = concordance_stratified(t_te, e_te, risk, strata_arr)
    auroc_3 = compute_auroc_at_time_stratified(t_te, e_te, risk, strata_arr, time_3yr)
    auroc_5 = compute_auroc_at_time_stratified(t_te, e_te, risk, strata_arr, time_5yr)
    eo_3 = expected_observed_ratio_at_time_stratified(ep3, t_te, e_te, strata_arr, time_3yr)
    eo_5 = expected_observed_ratio_at_time_stratified(ep5, t_te, e_te, strata_arr, time_5yr)

    return {
        "c_index": ci,
        "auroc_3yr": auroc_3,
        "auroc_5yr": auroc_5,
        "eo_ratio_3yr": eo_3,
        "eo_ratio_5yr": eo_5,
        "risk_scores": risk,
        "event_prob_3yr": ep3,
        "event_prob_5yr": ep5,
    }

In [ ]:
# -----------------------------------------------------------------------------
# Load and harmonize inputs
# -----------------------------------------------------------------------------
cohort = pd.read_csv(cohort_path)
outcomes = pd.read_csv(outcomes_path)
pcp_hf_covars = load_pcp_hf_features(covariates_path)
cmr = load_cmr_marker_features(cmr_markers_path)
aux = load_auxiliary_features(covariates_path)
latents = load_latent_features(latents_path)

split_df = cohort[["case_id", "split"]]
analysis_df = split_df.merge(outcomes, on="case_id", how="left")
analysis_df = analysis_df.merge(pcp_hf_covars, on="case_id", how="left")
analysis_df = analysis_df.merge(cmr, on="case_id", how="left")
# "age" and "sex" are already in pcp_hf_covars
unique_aux_cols = list(aux.columns.difference(pcp_hf_covars.columns))
analysis_df = analysis_df.merge(aux[["case_id"] + unique_aux_cols], on="case_id", how="left")
analysis_df = analysis_df.merge(latents, on="case_id", how="left")

analysis_df = prepare_survival_targets(analysis_df, endpoint=endpoint_col)

pcp_hf_cols = sorted([c for c in pcp_hf_covars.columns if c != "case_id"])
latent_cols = sorted(
    [c for c in analysis_df.columns if re.match(r"^x\d+$", c)],
    key=lambda x: int(x[1:]),
)
if len(latent_cols) == 0:
    raise RuntimeError("No latent columns found (expected x0, x1, ...).")
# Since we fit a sex-stratified model, we don't need to include sex as a predictor
cmr_marker_cols = sorted([c for c in cmr.columns if c != "case_id"])
cmr_score_cols = cmr_marker_cols + ["age", "bsa"]

required_cols = ["split", "time", "event"] + pcp_hf_cols + cmr_score_cols + latent_cols
analysis_df = analysis_df.dropna(subset=required_cols).reset_index(drop=True)

df_train = analysis_df.loc[analysis_df["split"] == "train"].reset_index(drop=True)
df_test = analysis_df.loc[analysis_df["split"] == "test"].reset_index(drop=True)

if len(df_train) == 0 or len(df_test) == 0:
    raise RuntimeError(f"Empty split after filtering: train={len(df_train)}, test={len(df_test)}")

print(f"Train: n={len(df_train)}, events={int(df_train['event'].sum())}")
print(f"Test : n={len(df_test)}, events={int(df_test['event'].sum())}")
print(f"PCP-HF columns used: {pcp_hf_cols}")
print(f"Latent dimensions: {len(latent_cols)}")
print(f"CMR marker columns used: {cmr_marker_cols}")
print(f"CMR score columns used: {cmr_score_cols}")

In [ ]:
# -----------------------------------------------------------------------------
# Step 1: derive scalar latent and CMR risk scores (train-fitted)
# -----------------------------------------------------------------------------

t_tr = df_train["time"].to_numpy(np.float64)
e_tr = df_train["event"].to_numpy(np.int64)
sex_tr = df_train["sex"].to_numpy(np.float64)

# 1A) Latent score: spline-expanded latent features + sex stratification
latent_spline_train, latent_design_infos = spline_expand_train(df_train[latent_cols], latent_cols, df_spline=3)
latent_spline_test = spline_expand_test(df_test[latent_cols], latent_cols, latent_design_infos)
expanded_latent_cols = list(latent_spline_train.columns)

best_pen_latent = tune_cox_penalizer(
    X_tr=latent_spline_train.to_numpy(np.float64),
    t_tr=t_tr,
    e_tr=e_tr,
    feature_cols=expanded_latent_cols,
    penalizer_grid=penalizer_grid,
    n_splits=n_splits,
    seed=seed,
    strata_arr=sex_tr,
)

latent_train_df = pd.concat(
    [
        latent_spline_train.reset_index(drop=True),
        df_train[["time", "event"]].reset_index(drop=True),
    ],
    axis=1,
)

cph_latent = _fit_cox(
    df_train=latent_train_df,
    feature_cols=expanded_latent_cols,
    penalizer=best_pen_latent,
    strata_arr=sex_tr,
)

latent_pred_train = latent_spline_train.copy()
latent_pred_train["_strata"] = df_train["sex"].to_numpy()
latent_pred_test = latent_spline_test.copy()
latent_pred_test["_strata"] = df_test["sex"].to_numpy()

latent_risk_train = cph_latent.predict_log_partial_hazard(latent_pred_train).to_numpy(np.float64)
latent_risk_test = cph_latent.predict_log_partial_hazard(latent_pred_test).to_numpy(np.float64)

# 1B) CMR score: 7 CMR markers + [age, sex, bsa]
cmr_spline_train, design_infos = spline_expand_train(df_train[cmr_score_cols], cmr_score_cols, df_spline=3)
cmr_spline_test = spline_expand_test(df_test[cmr_score_cols], cmr_score_cols, design_infos)
expanded_cmr_cols = list(cmr_spline_train.columns)

best_pen_cmr = tune_cox_penalizer(
    X_tr=cmr_spline_train.to_numpy(np.float64),
    t_tr=t_tr,
    e_tr=e_tr,
    feature_cols=expanded_cmr_cols,
    penalizer_grid=penalizer_grid,
    n_splits=n_splits,
    seed=seed,
    strata_arr=sex_tr,
)

cmr_train_df = pd.concat(
    [
        cmr_spline_train.reset_index(drop=True),
        df_train[["time", "event"]].reset_index(drop=True),
    ],
    axis=1,
)

cph_cmr = _fit_cox(
    df_train=cmr_train_df,
    feature_cols=expanded_cmr_cols,
    penalizer=best_pen_cmr,
    strata_arr=sex_tr,
)

cmr_pred_train = cmr_spline_train.copy()
cmr_pred_train["_strata"] = df_train["sex"].to_numpy()
cmr_pred_test = cmr_spline_test.copy()
cmr_pred_test["_strata"] = df_test["sex"].to_numpy()
cmr_risk_train = cph_cmr.predict_log_partial_hazard(cmr_pred_train).to_numpy(np.float64)
cmr_risk_test = cph_cmr.predict_log_partial_hazard(cmr_pred_test).to_numpy(np.float64)

df_train_scored = df_train.copy()
df_test_scored = df_test.copy()
df_train_scored["latent_risk_score"] = latent_risk_train
df_test_scored["latent_risk_score"] = latent_risk_test
df_train_scored["cmr_risk_score"] = cmr_risk_train
df_test_scored["cmr_risk_score"] = cmr_risk_test

print(f"[Step 1] Best penalizer (latent score model): {best_pen_latent}")
print(f"[Step 1] Best penalizer (CMR score model): {best_pen_cmr}")

In [ ]:
# -----------------------------------------------------------------------------
# Step 2: add each score to PCP-HF covariates and evaluate on test set
# -----------------------------------------------------------------------------
model_specs = {
    "pcp_hf_plus_latent_score": pcp_hf_cols + ["latent_risk_score"],
    "pcp_hf_plus_cmr_score": pcp_hf_cols + ["cmr_risk_score"],
}

results = {}
penalizers_step2 = {}
calibration_payload = {}

for model_name, feature_cols in model_specs.items():
    X_tr = df_train_scored[feature_cols].to_numpy(np.float64)

    best_pen = tune_cox_penalizer(
        X_tr=X_tr,
        t_tr=t_tr,
        e_tr=e_tr,
        feature_cols=feature_cols,
        penalizer_grid=penalizer_grid,
        n_splits=n_splits,
        seed=seed,
        strata_arr=sex_tr,
    )

    cph = _fit_cox(
        df_train=df_train_scored,
        feature_cols=feature_cols,
        penalizer=best_pen,
        strata_arr=sex_tr,
    )

    eval_out = _evaluate_model(
        cph=cph,
        df_test=df_test_scored,
        feature_cols=feature_cols,
        time_3yr=time_3yr,
        time_5yr=time_5yr,
        strata_arr=df_test["sex"].to_numpy(),
    )

    results[model_name] = {
        "c_index": eval_out["c_index"],
        "auroc_3yr": eval_out["auroc_3yr"],
        "auroc_5yr": eval_out["auroc_5yr"],
        "eo_ratio_3yr": eval_out["eo_ratio_3yr"],
        "eo_ratio_5yr": eval_out["eo_ratio_5yr"],
    }
    penalizers_step2[model_name] = best_pen

    calibration_payload[model_name] = {
        "time_3yr": {
            "event_prob": eval_out["event_prob_3yr"].tolist(),
            "times": df_test_scored["time"].to_list(),
            "events": df_test_scored["event"].to_list(),
            "risk_scores": eval_out["risk_scores"].tolist(),
        },
        "time_5yr": {
            "event_prob": eval_out["event_prob_5yr"].tolist(),
            "times": df_test_scored["time"].to_list(),
            "events": df_test_scored["event"].to_list(),
            "risk_scores": eval_out["risk_scores"].tolist(),
        },
    }

results_df = pd.DataFrame(results).T
print("\n[Step 2] Best penalizers:")
for k, v in penalizers_step2.items():
    print(f"  {k}: {v}")

print("\n[Step 2] Test-set metrics:")
results_df